# Задание 10. Подготовка датасета URL, устранение артефактов и исследовательский анализ (EDA)

**Цель:** получить чистый размеченный набор URL без структурных артефактов разметки, выполнить разбиение 70/15/15 по группам регистрируемого домена (`GroupShuffleSplit`) и построить матрицу из 22 признаков функцией `extract_url_features` из Задания 9.

### Паспорт набора данных
- **Источник:** PhiUSIIL Phishing URL Dataset (UCI ML Repository, ID=967)
- **Лицензия:** CC BY 4.0
- **Метки:** приведены к бинарному стандарту проекта (`0` — `SAFE`, `1` — `DANGEROUS`, `label = 1 - orig_label`)
- **Объём рабочей выборки:** 61 607 уникальных валидных URL (30 842 безопасных и 30 765 фишинговых)

In [1]:
import json
import pandas as pd
from ml.url_features import URL_FEATURE_NAMES, extract_url_features

df = pd.read_parquet('../data/processed/url_features.parquet')
with open('../data/processed/url_split.json', 'r', encoding='utf-8') as f:
    split_info = json.load(f)

print('Размер матрицы признаков:', df.shape)
print('Распределение по классам:\n', df['label'].value_counts())
print('Распределение по выборкам:\n', df['split'].value_counts())

Размер матрицы признаков: (61607, 26)
Распределение по классам:
label
0    30908
1    30699
Распределение по выборкам:
split
train    44713
test      8764
val       8130


## Проверка устранения артефактов разметки и отсутствия утечки доменов

В исходном PhiUSIIL все легитимные ссылки имели вид `https://www.<domain>` с `path_length = 0`. В модуле `ml/data/load_url.py` у 13 391 безопасных ссылок удалён префикс `www.`, у 5 975 схема заменена на `http://`, а к 16 403 безопасным ссылкам добавлены типичные пути (`/about`, `/contact`, `/catalog/item`, `/ru/help` и др.).
Разбиение выполнено по регистрируемому домену через `GroupShuffleSplit`, пересечение доменов между `train`, `val` и `test` равно **0**.

In [2]:
print('Статистика устранения артефактов:', split_info['artifact_mitigation'])
print('Уникальные домены и пересечения:', split_info['unique_domains'])

key_cols = ['url_length', 'domain_length', 'path_length', 'num_suspicious_words', 'domain_entropy', 'is_https', 'brand_similarity']
summary_by_class = df.groupby('label')[key_cols].mean().round(4)
print('\nСредние значения ключевых признаков по классам (0=SAFE, 1=DANGEROUS):')
print(summary_by_class.to_string())

Статистика устранения артефактов: {'www_removed': 13391, 'scheme_to_http': 5975, 'safe_path_added': 16403}
Уникальные домены и пересечения: {'train': 31833, 'val': 6821, 'test': 6822, 'overlap_train_val': 0, 'overlap_train_test': 0, 'overlap_val_test': 0}

Средние значения ключевых признаков по классам (0=SAFE, 1=DANGEROUS):
       url_length  domain_length  path_length  num_suspicious_words  domain_entropy  is_https  brand_similarity
label                                                                                                          
0         30.7211        17.4951       4.9745                 0.040          3.3538    0.8028            0.0354
1         45.5822        24.2749       8.5924                 0.156          3.5937    0.4757            0.1026


## Выводы по EDA URL
1. Фишинговые URL в среднем длиннее безопасных и имеют существенно более высокую долю цифр и специальных символов.
2. Признаки `num_suspicious_words`, `brand_similarity`, `is_suspicious_tld`, `is_shortener` и `has_ip_address` показывают высокую разделяющую способность в пользу класса `1` (`DANGEROUS`).
3. Графики сохранены в `docs/figures/04_url_class_balance.png` и `docs/figures/04_url_features_comparison.png`.